# Super Outbreak: 500-mb geopotential height

The field is the NOAA/NCEP–NCAR Reanalysis 1 four-times-daily geopotential height at the 500-mb pressure level, valid 1200 GMT on April 3, 1974. The notebook requests only this time, level, and North American domain through NOAA PSL's NetCDF Subset Service. Values remain in metres; only the contour labels use the contemporary upper-air-chart decametre convention.


In [ ]:
from datetime import datetime, timedelta
import json
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np
from scipy.io import netcdf_file

DATA_URL = (
    "https://psl.noaa.gov/thredds/ncss/grid/Datasets/ncep.reanalysis/"
    "pressure/hgt.1974.nc?var=hgt&north=60&west=220&east=300&south=20"
    "&horizStride=1&time=1974-04-03T12%3A00%3A00Z"
    "&vertCoord=500&accept=netcdf3"
)
COASTLINE_URL = (
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/"
    "master/geojson/ne_110m_coastline.geojson"
)
cache = Path("data")
cache.mkdir(exist_ok=True)


def download(url, destination):
    if not destination.exists():
        request = Request(url, headers={"User-Agent": "lab74 showcase"})
        with urlopen(request) as response, destination.open("wb") as stream:
            stream.write(response.read())
    return destination


def load_geopotential_height():
    source = download(
        DATA_URL,
        cache / "ncep_reanalysis_geopotential_height_500mb_19740403_1200.nc",
    )
    with netcdf_file(source, mmap=False) as dataset:
        level = float(dataset.variables["level"][0])
        time = float(dataset.variables["time"][0])
        latitude = dataset.variables["lat"][:].copy()
        longitude = dataset.variables["lon"][:].copy()
        geopotential_height = dataset.variables["hgt"][0, 0].copy()
        units = dataset.variables["hgt"].units.decode()
        field_name = dataset.variables["hgt"].var_desc.decode()

    valid_time = datetime(1800, 1, 1) + timedelta(hours=time)
    assert level == 500 and units == "m"
    assert field_name == "Geopotential height"
    assert valid_time == datetime(1974, 4, 3, 12)

    longitude = np.where(longitude > 180, longitude - 360, longitude)
    lat_order = np.argsort(latitude)
    lon_order = np.argsort(longitude)
    return (
        longitude[lon_order],
        latitude[lat_order],
        geopotential_height[np.ix_(lat_order, lon_order)],
    )


def load_linework(url, filename):
    source = download(url, cache / filename)
    with source.open() as stream:
        features = json.load(stream)["features"]
    for feature in features:
        geometry = feature["geometry"]
        coordinates = geometry["coordinates"]
        if geometry["type"] == "LineString":
            yield np.asarray(coordinates)
        elif geometry["type"] == "MultiLineString":
            yield from (np.asarray(line) for line in coordinates)


longitude, latitude, geopotential_height = load_geopotential_height()
coastline = list(load_linework(COASTLINE_URL, "ne_110m_coastline.geojson"))



In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import lab74

lab74.use(accent=None)
out = Path("output")
out.mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(7.2, 5.2))
extent = (-127.5, -67.5, 24, 55.5)
ax.set(xlim=extent[:2], ylim=extent[2:])
ax.set_aspect(1 / np.cos(np.deg2rad(40)))

longitudes = np.arange(-120, -69, 10)
latitudes = np.arange(30, 51, 10)
lab74.format_graticule(ax, longitudes, latitudes, draw_lines=False)

lab74.map_linework(ax, coastline)

domain = (
    (longitude >= extent[0]) & (longitude <= extent[1]),
    (latitude >= extent[2]) & (latitude <= extent[3]),
)
visible_height = geopotential_height[np.ix_(domain[1], domain[0])]
first_level = np.floor(visible_height.min() / 60) * 60
last_level = np.ceil(visible_height.max() / 60) * 60
height_levels = np.arange(first_level, last_level + 1, 60)
height_contours, _ = lab74.technical_contour(
    ax,
    longitude,
    latitude,
    geopotential_height,
    levels=height_levels,
    labels=True,
    label_format=lambda value: f"{value / 10:.0f}",
    label_kwargs={
        "levels": height_levels[::2],
        "fontsize": 7,
        "inline_spacing": 3,
    },
    linewidths=0.7,
    zorder=2,
)

local_minima = []
for row in range(1, len(latitude) - 1):
    for column in range(1, len(longitude) - 1):
        if not (30 <= latitude[row] <= 48 and -112 <= longitude[column] <= -88):
            continue
        neighborhood = geopotential_height[row - 1 : row + 2, column - 1 : column + 2]
        if geopotential_height[row, column] == neighborhood.min():
            local_minima.append((geopotential_height[row, column], row, column))
if local_minima:
    _, row, column = min(local_minima)
    ax.text(
        longitude[column], latitude[row], "L",
        ha="center", va="center", fontsize=11, zorder=4,
    )

lab74.plate_label(
    ax, "500 MB\nAPRIL 3, 1974\n1200 GMT",
    bbox={"facecolor": lab74.PAPER, "edgecolor": "none"}, zorder=5,
)

fig.savefig(out / "04_super_outbreak_500mb_geopotential_height.png")
plt.close(fig)
